In [1]:
# NativeNext management setup; original course cells follow unchanged.
import sys
sys.path.insert(0,'C:\\Users\\[LEARNER_NAME_OMITTED]\\.v\\hi\\20261019T155611-ae6c3405')
import next_identity as _identity
_identity.write_child_runtime_identity()
import next_adapter as _next
_next.install('C:\\Users\\[LEARNER_NAME_OMITTED]\\.v\\hi\\20261019T155611-ae6c3405\\source\\HaiLong_Colab_Exercises','C:\\Users\\[LEARNER_NAME_OMITTED]\\.v\\hi\\20261019T155611-ae6c3405','04_fastai_part2/12_accel_sgd.ipynb')


# Google Colab edition

Run the setup cell first, then work from top to bottom. Keep the course ZIP in My Drive. Use a fresh Colab runtime when switching dependency profiles. **Select a GPU runtime before running this notebook.** 

Original numbered course notebook, adapted for Colab. Run in numerical order: later lessons load models trained by earlier lessons. Use the same Drive result folder across sessions. W&B logs stay offline. Full training can exceed a free Colab session; QUICK_RUN does not shorten these original Part 2 experiments.

Source: https://github.com/fastai/course22p2/blob/df9323235bc395b5c2f58a3d08b83761947b9b93/nbs/12_accel_sgd.ipynb

In [2]:
# Colab setup. Run this cell before the exercise cells.
from pathlib import Path
import os, sys, zipfile

USE_GOOGLE_DRIVE = True  #@param {type:"boolean"}
QUICK_RUN = False  #@param {type:"boolean"}
ARCHIVE_PATH = "/content/drive/MyDrive/HaiLong_Colab_Exercises_2026-10-03.zip"  #@param {type:"string"}

try:
    from google.colab import drive, files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if USE_GOOGLE_DRIVE:
        if not Path('/content/drive/MyDrive').is_dir():
            drive.mount('/content/drive', timeout_ms=300000)
    COURSE_ROOT = Path('/content/HaiLong_Colab_Exercises')
    if not (COURSE_ROOT / 'colab_runtime.py').is_file():
        archive = Path(ARCHIVE_PATH)
        if not archive.is_file():
            print('Select HaiLong_Colab_Exercises_2026-10-03.zip from your computer.')
            uploaded = files.upload()
            choices = [Path(n) for n in uploaded if n.endswith('.zip')]
            if len(choices) != 1:
                raise ValueError('Upload exactly one course ZIP file.')
            archive = choices[0]
        with zipfile.ZipFile(archive) as z:
            base = Path('/content').resolve()
            for member in z.infolist():
                target = (base / member.filename).resolve()
                if not target.is_relative_to(base) or not member.filename.startswith('HaiLong_Colab_Exercises/'):
                    raise ValueError('This is not the expected course archive.')
            z.extractall(base)
    if not (COURSE_ROOT / 'colab_runtime.py').is_file():
        raise FileNotFoundError('The ZIP must contain HaiLong_Colab_Exercises/colab_runtime.py.')
else:
    # Also permits local checks without assuming the current notebook directory.
    COURSE_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                        if (p / 'colab_runtime.py').is_file()), None)
    if COURSE_ROOT is None:
        raise RuntimeError('Open this notebook in Colab or run it inside the extracted course folder.')

os.environ['COURSE_SMOKE'] = '1' if QUICK_RUN else '0'
os.environ['WANDB_MODE'] = 'offline'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
sys.path.insert(0, str(COURSE_ROOT))
from colab_runtime import setup_course, finish_run
# Filled individually for each notebook by the builder.

COLAB_CONTEXT = setup_course(COURSE_ROOT, '04_fastai_part2', 'part2', '12_accel_sgd.ipynb', USE_GOOGLE_DRIVE, install=IN_COLAB)

Exercise folder: C:\Users\[LOCAL_USER]\.v\hi\20261019T155611-ae6c3405\source\HaiLong_Colab_Exercises\04_fastai_part2
Run mode: FULL EXERCISE
Results: C:\Users\[LOCAL_USER]\.v\hi\20261019T155611-ae6c3405\source\HaiLong_Colab_Exercises\04_fastai_part2\results


## Full exercise evidence

- Run all exercise cells top to bottom in hosted Colab and preserve their outputs, errors and execution counts.
- Keep QUICK_RUN=False; shortened checks or synthetic replacements for real datasets do not count as full original exercises.
- Preserve plots, models, metrics and run configurations in Drive; a final-cell marker alone is insufficient.
- Retain original datasets, model definitions, training schedules and instructional comparisons.
- Run prerequisite checkpoint-producing notebooks first; do not replace missing weights with untrained models.

- Review the labeled authored objective answers alongside genuine saved exercise outputs; prepared prose does not certify hosted execution, personal reflection or independent learner recall.

This cell describes evidence to collect; it does not certify completion. GPU batch caps preserve dataset coverage and epoch counts, but are documented configuration changes.


In [3]:
MAX_BATCH_SIZE = 32  #@param {type:"integer"}
# Increase on a larger GPU to match the original batch sizes.
assert MAX_BATCH_SIZE > 0

In [4]:
#|default_exp sgd

# Accelerated SGD

In [5]:
#|export
import torch

from miniai.datasets import *
from miniai.conv import *
from miniai.learner import *
from miniai.activations import *
from miniai.init import *

In [6]:
import pickle,gzip,math,os,time,shutil,torch,matplotlib as mpl,numpy as np,matplotlib.pyplot as plt
import fastcore.all as fc
from collections.abc import Mapping
from pathlib import Path
from operator import attrgetter,itemgetter
from functools import partial
from copy import copy
from contextlib import contextmanager

import torchvision.transforms.functional as TF,torch.nn.functional as F
from torch import tensor,nn,optim
from torch.utils.data import DataLoader,default_collate
from torch.nn import init
from torch.optim import lr_scheduler
from torcheval.metrics import MulticlassAccuracy
from datasets import load_dataset,load_dataset_builder

from miniai.datasets import *
from miniai.conv import *
from miniai.learner import *
from miniai.activations import *
from miniai.init import *

In [7]:
from fastcore.test import test_close

torch.set_printoptions(precision=2, linewidth=140, sci_mode=False)
torch.manual_seed(1)

import logging
logging.disable(logging.WARNING)

set_seed(42)

In [8]:
xl,yl = 'image','label'
name = "zalando-datasets/fashion_mnist"
dsd = load_dataset(name)

bs = min(1024, MAX_BATCH_SIZE)
xmean,xstd = 0.28, 0.35

@inplace
def transformi(b): b[xl] = [(TF.to_tensor(o)-xmean)/xstd for o in b[xl]]

tds = dsd.with_transform(transformi)
dls = DataLoaders.from_dd(tds, bs, num_workers=4)

README.md: 0.00B [00:00, ?B/s]

C:\Users\[LOCAL_USER]\.v\hlimg31\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\[LOCAL_USER]\.v\hi\20261019T155611-ae6c3405\private_cache\hf\hub\datasets--zalando-datasets--fashion_mnist. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


train-00000-of-00001.parquet:   0%|          | 0.00/30.9M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/5.18M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/60000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [9]:
metrics = MetricsCB(accuracy=MulticlassAccuracy())
astats = ActivationStats(fc.risinstance(GeneralRelu))
cbs = [DeviceCB(), metrics, ProgressCB(plot=True), astats]
act_gr = partial(GeneralRelu, leak=0.1, sub=0.4)
iw = partial(init_weights, leaky=0.1)
lrf_cbs = [DeviceCB(), LRFinderCB()]

## Optimizers

### SGD

In [10]:
class SGD:
    def __init__(self, params, lr, wd=0.):
        params = list(params)
        fc.store_attr()
        self.i = 0

    def step(self):
        with torch.no_grad():
            for p in self.params:
                self.reg_step(p)
                self.opt_step(p)
        self.i +=1

    def opt_step(self, p): p -= p.grad * self.lr
    def reg_step(self, p):
        if self.wd != 0: p *= 1 - self.lr*self.wd

    def zero_grad(self):
        for p in self.params: p.grad.data.zero_()

In [11]:
set_seed(42)
model = get_model(act_gr, norm=nn.BatchNorm2d).apply(iw)
learn = TrainLearner(model, dls, F.cross_entropy, lr=0.4, cbs=cbs, opt_func=SGD)

In [12]:
learn.fit(3)

Storage check (before_fit): artifacts 0.000 GB; configured budget 80.000 GB; additional estimate 0.035 GB; runtime free 524.210 GB. Budget is not Drive quota.
Storage check (checkpoint_write): artifacts 0.000 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 524.210 GB. Budget is not Drive quota.


accuracy,loss,epoch,train
0.839,0.440,0,train
0.866,0.356,0,eval
0.882,0.320,1,train
0.880,0.332,1,eval
0.893,0.287,2,train
0.884,0.317,2,eval


Storage check (checkpoint_write): artifacts 0.000 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 524.210 GB. Budget is not Drive quota.


<Figure size 600x400 with 1 Axes>

Storage check (checkpoint_write): artifacts 0.000 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 524.209 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 524.209 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 524.209 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 524.208 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 524.208 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 524.208 GB. Budget is not Drive quota.


Consider the difference between *weight decay* and *L2 regularization*:

``` python
weight -= lr*wd*weight
```

...vs...

``` python
weight.grad += wd*weight
```

## Authored explanation: weight decay and L2

For ordinary SGD with loss gradient `g`, adding the penalty `(wd/2)*sum(w**2)` adds `wd*w` to the gradient. The update is therefore `w_new = w - lr*(g + wd*w) = (1-lr*wd)*w - lr*g`. Applying the two operations separately gives the same update when both use the same pre-update weights, learning rate and coefficient, with no momentum or adaptive gradient transformation. A penalty written without the factor `1/2` has twice this derivative, so coefficient conventions matter.

The distinction becomes material with momentum and adaptive optimizers. Adding `wd*w` to `weight.grad` feeds that term through the optimizer's accumulated or scaled gradient. Direct weight decay shrinks the weights independently of those gradient statistics. In this notebook's original `SGD.step`, `reg_step` shrinks each parameter before `opt_step`; its `Momentum` subclass smooths the loss gradient while keeping that separate shrinkage. These are not generally the same as injecting an L2 penalty into the momentum gradient, and AdamW deliberately decouples shrinkage from Adam's adaptive gradient update.

Use one intended regularization method rather than accidentally applying both. This is a derivation from the preserved optimizer code, not a claim that either method won an experiment. [Pinned course source, original cell 13](https://github.com/fastai/course22p2/blob/df9323235bc395b5c2f58a3d08b83761947b9b93/nbs/12_accel_sgd.ipynb), [Decoupled Weight Decay Regularization](https://arxiv.org/abs/1711.05101), [PyTorch SGD](https://docs.pytorch.org/docs/2.14/generated/torch.optim.SGD.html), [AdamW](https://docs.pytorch.org/docs/2.14/generated/torch.optim.AdamW.html).


### Momentum

In [13]:
xs = torch.linspace(-4, 4, 100)
ys = 1 - (xs/3) ** 2 + torch.randn(100) * 0.1

In [14]:
_,axs = plt.subplots(2,2, figsize=(12,8))
betas = [0.5,0.7,0.9,0.99]
for beta,ax in zip(betas, axs.flatten()):
    ax.scatter(xs,ys)
    avg,res = 0,[]
    for yi in ys:
        avg = beta*avg + (1-beta)*yi
        res.append(avg)
    ax.plot(xs,np.array(res), color='red');
    ax.set_title(f'beta={beta}')

In [15]:
class Momentum(SGD):
    def __init__(self, params, lr, wd=0., mom=0.9):
        super().__init__(params, lr=lr, wd=wd)
        self.mom=mom

    def opt_step(self, p):
        if not hasattr(p, 'grad_avg'): p.grad_avg = torch.zeros_like(p.grad)
        p.grad_avg = p.grad_avg*self.mom + p.grad*(1-self.mom)
        p -= self.lr * p.grad_avg

In [16]:
set_seed(42)
model = get_model(act_gr, norm=nn.BatchNorm2d).apply(iw)
learn = TrainLearner(model, dls, F.cross_entropy, lr=1.5, cbs=cbs, opt_func=Momentum)

In [17]:
learn.fit(3)

Storage check (before_fit): artifacts 0.000 GB; configured budget 80.000 GB; additional estimate 0.036 GB; runtime free 524.187 GB. Budget is not Drive quota.
Storage check (checkpoint_write): artifacts 0.000 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 524.187 GB. Budget is not Drive quota.


accuracy,loss,epoch,train
0.822,0.504,0,train
0.858,0.392,0,eval
0.863,0.376,1,train
0.854,0.413,1,eval
0.875,0.344,2,train
0.869,0.355,2,eval


Storage check (checkpoint_write): artifacts 0.000 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 524.185 GB. Budget is not Drive quota.


<Figure size 600x400 with 1 Axes>

Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 524.059 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 524.051 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 524.051 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 524.049 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 524.049 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 524.049 GB. Budget is not Drive quota.


In [18]:
astats.color_dim()

### RMSProp

In [19]:
class RMSProp(SGD):
    def __init__(self, params, lr, wd=0., sqr_mom=0.99, eps=1e-5):
        super().__init__(params, lr=lr, wd=wd)
        self.sqr_mom,self.eps = sqr_mom,eps

    def opt_step(self, p):
        if not hasattr(p, 'sqr_avg'): p.sqr_avg = p.grad**2
        p.sqr_avg = p.sqr_avg*self.sqr_mom + p.grad**2*(1-self.sqr_mom)
        p -= self.lr * p.grad/(p.sqr_avg.sqrt() + self.eps)

In [20]:
set_seed(42)
model = get_model(act_gr, norm=nn.BatchNorm2d).apply(iw)
learn = TrainLearner(model, dls, F.cross_entropy, lr=3e-3, cbs=cbs, opt_func=RMSProp)
learn.fit(3)

Storage check (before_fit): artifacts 0.000 GB; configured budget 80.000 GB; additional estimate 0.035 GB; runtime free 524.028 GB. Budget is not Drive quota.
Storage check (checkpoint_write): artifacts 0.000 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 524.028 GB. Budget is not Drive quota.


accuracy,loss,epoch,train
0.845,0.423,0,train
0.870,0.356,0,eval
0.884,0.315,1,train
0.881,0.328,1,eval
0.896,0.280,2,train
0.888,0.308,2,eval


Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 524.028 GB. Budget is not Drive quota.


<Figure size 600x400 with 1 Axes>

Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 524.028 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 524.029 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 524.029 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 524.028 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 524.028 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 524.028 GB. Budget is not Drive quota.


In [21]:
astats.color_dim()

### Adam

In [22]:
class Adam(SGD):
    def __init__(self, params, lr, wd=0., beta1=0.9, beta2=0.99, eps=1e-5):
        super().__init__(params, lr=lr, wd=wd)
        self.beta1,self.beta2,self.eps = beta1,beta2,eps

    def opt_step(self, p):
        if not hasattr(p, 'avg'): p.avg = torch.zeros_like(p.grad.data)
        if not hasattr(p, 'sqr_avg'): p.sqr_avg = torch.zeros_like(p.grad.data)
        p.avg = self.beta1*p.avg + (1-self.beta1)*p.grad
        unbias_avg = p.avg / (1 - (self.beta1**(self.i+1)))
        p.sqr_avg = self.beta2*p.sqr_avg + (1-self.beta2)*(p.grad**2)
        unbias_sqr_avg = p.sqr_avg / (1 - (self.beta2**(self.i+1)))
        p -= self.lr * unbias_avg / (unbias_sqr_avg + self.eps).sqrt()

In [23]:
set_seed(42)
model = get_model(act_gr, norm=nn.BatchNorm2d).apply(iw)
learn = TrainLearner(model, dls, F.cross_entropy, lr=6e-3, cbs=cbs, opt_func=Adam)
learn.fit(3)

Storage check (before_fit): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.035 GB; runtime free 524.007 GB. Budget is not Drive quota.
Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 524.007 GB. Budget is not Drive quota.


accuracy,loss,epoch,train
0.846,0.425,0,train
0.873,0.359,0,eval
0.883,0.316,1,train
0.858,0.385,1,eval
0.896,0.283,2,train
0.883,0.318,2,eval


Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 524.007 GB. Budget is not Drive quota.


<Figure size 600x400 with 1 Axes>

Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 524.006 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.002 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 524.006 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.002 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 524.006 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.002 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 524.005 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.002 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 524.005 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.002 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 524.005 GB. Budget is not Drive quota.


## Schedulers

We've already seen how we can easily write a custom LR-adjusting callback or `Learner`, or can use the predefined PyTorch schedulers. We'll use the predefined ones for now since there's nothing new to learn in implementing them ourselves.

In [24]:
' '.join(o for o in dir(lr_scheduler) if o[0].isupper() and o[1].islower())

'Any Callable ChainedScheduler ConstantLR CosineAnnealingLR CosineAnnealingWarmRestarts Counter CyclicLR ExponentialLR LambdaLR LinearLR Literal MultiStepLR MultiplicativeLR OneCycleLR Optimizer Optional PolynomialLR ReduceLROnPlateau Self SequentialLR StepLR SupportsFloat Tensor TypedDict Union'

In [25]:
' '.join(filter(lambda x: x[0].isupper() and x[1].islower(), dir(lr_scheduler)))

'Any Callable ChainedScheduler ConstantLR CosineAnnealingLR CosineAnnealingWarmRestarts Counter CyclicLR ExponentialLR LambdaLR LinearLR Literal MultiStepLR MultiplicativeLR OneCycleLR Optimizer Optional PolynomialLR ReduceLROnPlateau Self SequentialLR StepLR SupportsFloat Tensor TypedDict Union'

In [26]:
learn = TrainLearner(get_model(), dls, F.cross_entropy, lr=6e-3, cbs=[DeviceCB(), SingleBatchCB()])
learn.fit(1)

In [27]:
opt = learn.opt
' '.join(o for o in dir(opt) if o[0]!='_')

'OptimizerPostHook OptimizerPreHook add_param_group defaults load_state_dict param_groups profile_hook_step register_load_state_dict_post_hook register_load_state_dict_pre_hook register_state_dict_post_hook register_state_dict_pre_hook register_step_post_hook register_step_pre_hook state state_dict step zero_grad'

In [28]:
opt

SGD (
Parameter Group 0
    dampening: 0
    differentiable: False
    foreach: None
    fused: None
    lr: 0.006
    maximize: False
    momentum: 0
    nesterov: False
    weight_decay: 0
)

In [29]:
param = next(iter(learn.model.parameters()))
st = opt.state[param]

In [30]:
st

{}

In [31]:
len(opt.param_groups)

1

In [32]:
pg = opt.param_groups[0]

In [33]:
list(pg)

['params',
 'lr',
 'momentum',
 'dampening',
 'weight_decay',
 'nesterov',
 'maximize',
 'foreach',
 'differentiable',
 'fused']

In [34]:
sched = lr_scheduler.CosineAnnealingLR(opt, 100)

In [35]:
sched.base_lrs

[0.006]

In [36]:
sched.get_last_lr()

[0.006]

In [37]:
def sched_lrs(sched, steps):
    lrs = [sched.get_last_lr()]
    for i in range(steps):
        sched.optimizer.step()
        sched.step()
        lrs.append(sched.get_last_lr())
    plt.plot(lrs)

In [38]:
sched_lrs(sched, 110)

### Scheduler callbacks

In [39]:
#|export
class BaseSchedCB(Callback):
    def __init__(self, sched): self.sched = sched
    def before_fit(self, learn): self.schedo = self.sched(learn.opt)
    def _step(self, learn):
        if learn.training: self.schedo.step()

In [40]:
#|export
class BatchSchedCB(BaseSchedCB):
    def after_batch(self, learn): self._step(learn)

In [41]:
#|export
class HasLearnCB(Callback):
    def before_fit(self, learn): self.learn = learn 
    def after_fit(self, learn): self.learn = None

In [42]:
#|export
class RecorderCB(Callback):
    def __init__(self, **d): self.d = d
    def before_fit(self, learn):
        self.recs = {k:[] for k in self.d}
        self.pg = learn.opt.param_groups[0]
    
    def after_batch(self, learn):
        if not learn.training: return
        for k,v in self.d.items():
            self.recs[k].append(v(self))

    def plot(self):
        for k,v in self.recs.items():
            plt.plot(v, label=k)
            plt.legend()
            plt.show()

In [43]:
def _lr(cb): return cb.pg['lr']

In [44]:
len(dls.train)

1875

In [45]:
tmax = 3 * len(dls.train)
sched = partial(lr_scheduler.CosineAnnealingLR, T_max=tmax)

In [46]:
set_seed(42)
model = get_model(act_gr, norm=nn.BatchNorm2d).apply(iw)
rec = RecorderCB(lr=_lr)
xtra = [BatchSchedCB(sched),rec]
learn = TrainLearner(model, dls, F.cross_entropy, lr=2e-2, cbs=cbs+xtra, opt_func=optim.AdamW)
learn.fit(3)

Storage check (before_fit): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.035 GB; runtime free 523.982 GB. Budget is not Drive quota.
Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 523.982 GB. Budget is not Drive quota.


accuracy,loss,epoch,train
0.833,0.464,0,train
0.867,0.368,0,eval
0.882,0.324,1,train
0.889,0.312,1,eval
0.909,0.246,2,train
0.902,0.272,2,eval


Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 523.980 GB. Budget is not Drive quota.


<Figure size 600x400 with 1 Axes>

Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 523.980 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.002 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 523.979 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.002 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 523.979 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.002 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 523.979 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.002 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 523.979 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.002 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 523.979 GB. Budget is not Drive quota.


In [47]:
rec.plot()

C:\Users\[LOCAL_USER]\.v\hi\20261019T155611-ae6c3405\private_cache\temp\ipykernel_37520\1338906730.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [48]:
#|export
class EpochSchedCB(BaseSchedCB):
    def after_epoch(self, learn): self._step(learn)

In [49]:
sched = partial(lr_scheduler.CosineAnnealingLR, T_max=3)
set_seed(42)
xtra = [EpochSchedCB(sched),rec]
model = get_model(act_gr, norm=nn.BatchNorm2d).apply(iw)
learn = TrainLearner(model, dls, F.cross_entropy, lr=2e-2, cbs=cbs+xtra, opt_func=optim.AdamW)
learn.fit(3)

Storage check (before_fit): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.035 GB; runtime free 523.960 GB. Budget is not Drive quota.
Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 523.960 GB. Budget is not Drive quota.


accuracy,loss,epoch,train
0.830,0.473,0,train
0.861,0.389,0,eval
0.872,0.351,1,train
0.871,0.362,1,eval
0.901,0.269,2,train
0.898,0.285,2,eval


Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 523.959 GB. Budget is not Drive quota.


<Figure size 600x400 with 1 Axes>

Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 523.959 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.002 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 523.958 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.002 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 523.958 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.002 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 523.958 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.002 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 523.958 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.002 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 523.958 GB. Budget is not Drive quota.


In [50]:
rec.plot()

C:\Users\[LOCAL_USER]\.v\hi\20261019T155611-ae6c3405\private_cache\temp\ipykernel_37520\1338906730.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 1cycle training

[Paper](https://arxiv.org/abs/1803.09820) by Leslie Smith.

In [51]:
def _beta1(cb): return cb.pg['betas'][0]
rec = RecorderCB(lr=_lr, mom=_beta1)

In [52]:
set_seed(42)
lr,epochs = 6e-2,5
model = get_model(act_gr, norm=nn.BatchNorm2d).apply(iw)
tmax = epochs * len(dls.train)
sched = partial(lr_scheduler.OneCycleLR, max_lr=lr, total_steps=tmax)
xtra = [BatchSchedCB(sched), rec]
learn = TrainLearner(model, dls, F.cross_entropy, lr=lr, cbs=cbs+xtra, opt_func=optim.AdamW)
learn.fit(epochs)

Storage check (before_fit): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.035 GB; runtime free 523.936 GB. Budget is not Drive quota.
Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 523.936 GB. Budget is not Drive quota.


accuracy,loss,epoch,train
0.819,0.506,0,train
0.833,0.509,0,eval
0.826,0.511,1,train
0.851,0.444,1,eval
0.850,0.427,2,train
0.843,0.423,2,eval
0.879,0.334,3,train
0.889,0.300,3,eval
0.909,0.246,4,train
0.903,0.267,4,eval


Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 523.936 GB. Budget is not Drive quota.


<Figure size 600x400 with 1 Axes>

Storage check (checkpoint_write): artifacts 0.002 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 523.936 GB. Budget is not Drive quota.


...

Storage check (checkpoint_write): artifacts 0.002 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 523.934 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.002 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 523.934 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.002 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 523.933 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.002 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 523.933 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.002 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 523.933 GB. Budget is not Drive quota.


In [53]:
rec.plot()

C:\Users\[LOCAL_USER]\.v\hi\20261019T155611-ae6c3405\private_cache\temp\ipykernel_37520\1338906730.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Export -

In [54]:
# The bundled miniai package is ready to import; no export is required.

Save your outputs

Run the following cell after finishing. Choose **File > Save** in Colab to preserve your code and displayed results. Files in `results/` and `models/` persist in Drive when enabled.

In [55]:
finish_run(COLAB_CONTEXT)

Saved run settings. Save the notebook in Colab to keep its cell outputs.


In [56]:
# NativeNext independent numerical probe.
_next.probe(globals())


__HAILONG_NATIVE_NEXT_FULL_EVIDENCE__{"contract": {"passed": true, "target": "04_fastai_part2/12_accel_sgd.ipynb", "ordinary_fits": 7, "ordinary_epochs": 23, "ordinary_train_batches": 43125, "ordinary_valid_batches": 3611, "single_batch_demonstrations": 1, "standalone_scheduler_demo_updates": 110}, "numerical_artifact_subprobe_only": true, "numerical_models": [{"cell_index": 15, "original_epochs": 3, "probe_rows": 64, "cross_entropy": 0.3774382472038269, "accuracy": 0.890625, "model_checkpoint_snapshot_equal": true, "exact_saved_inference_equal": true, "activation_statistics_retained": true, "optimizer_strict_roundtrip": true, "preserved_gradients_restored": true, "rng_validation": {"python_numpy_torch_cpu_generator_restore": true, "cuda_state_devices": 1, "cuda_validation": "saved ByteTensor structure only; no GPU generator replay claim"}}, {"cell_index": 23, "original_epochs": 3, "probe_rows": 64, "cross_entropy": 0.37968388199806213, "accuracy": 0.890625, "model_checkpoint_snapshot_

{'contract': {'passed': True,
  'target': '04_fastai_part2/12_accel_sgd.ipynb',
  'ordinary_fits': 7,
  'ordinary_epochs': 23,
  'ordinary_train_batches': 43125,
  'ordinary_valid_batches': 3611,
  'single_batch_demonstrations': 1,
  'standalone_scheduler_demo_updates': 110},
 'numerical_artifact_subprobe_only': True,
 'numerical_models': [{'cell_index': 15,
   'original_epochs': 3,
   'probe_rows': 64,
   'cross_entropy': 0.3774382472038269,
   'accuracy': 0.890625,
   'model_checkpoint_snapshot_equal': True,
   'exact_saved_inference_equal': True,
   'activation_statistics_retained': True,
   'optimizer_strict_roundtrip': True,
   'preserved_gradients_restored': True,
   'rng_validation': {'python_numpy_torch_cpu_generator_restore': True,
    'cuda_state_devices': 1,
    'cuda_validation': 'saved ByteTensor structure only; no GPU generator replay claim'}},
  {'cell_index': 23,
   'original_epochs': 3,
   'probe_rows': 64,
   'cross_entropy': 0.37968388199806213,
   'accuracy': 0.8906